# Phase 5 — Model Optimization
### American Express Default Prediction

**Phase 4 recap (treated as source of truth):** a controlled feature ablation found that
**"Latest + Historical Summary"** — `last` + `categorical_last` + `mean`/`std`/`min`/`max`,
852 features — matched or slightly beat the full 1,239-feature set on both model families:
LightGBM AMEX 0.7935 vs. 0.7918 (full), CatBoost AMEX 0.7939 vs. 0.7936 (full). Phase 5
formally adopts this 852-feature configuration and asks a narrower question:

**Central question:** how much additional predictive performance can be obtained through
targeted model optimization, once the feature representation has already been fixed?

**Scope, fixed in advance:** LightGBM and CatBoost only (Logistic Regression is not
tuned — it was never the strongest baseline and tuning a linear model rarely moves an
AMEX-metric comparison much). No exhaustive search: at most 6–8 new LightGBM experiments,
2–3 new CatBoost experiments, all sequential, all compared against the *existing* Phase 4
852-feature results — which are **cited, not retrained**, since re-running an identical
configuration on an identical split would only burn compute without adding information.


In [1]:
import os
import gc
import re
import time

import numpy as np
import pandas as pd
import psutil

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

DATA_DIR = "../data"
PROCESSED_PATH = os.path.join(DATA_DIR, "processed", "train_features.parquet")
PRED_DIR = os.path.join(DATA_DIR, "processed")  # already gitignored (data/)
RANDOM_SEED = 42

_process = psutil.Process(os.getpid())
def rss_gb():
    return _process.memory_info().rss / 1e9

PHASE5_START = time.time()
print(f"Starting RSS: {rss_gb():.2f} GB")


Starting RSS: 0.13 GB


## Setup: Reproducing the Phase 3/4 Split and Feature Machinery Exactly

Copied verbatim from Phase 4 (same exclusion list, same split call, same seed, same
feature-group classifier) — not a redesign. `data/processed/train_features.parquet` is
Phase 2's output and is not touched or recomputed.


In [2]:
t0 = time.time()
customer_features = pd.read_parquet(PROCESSED_PATH)
print(f"Loaded {customer_features.shape} in {time.time()-t0:.1f}s. RSS: {rss_gb():.2f} GB")

DROP_RAW_FAMILIES = ["D_87", "D_88", "D_108", "D_110", "D_111", "B_39", "D_73", "B_42"]
drop_cols = [
    c for c in customer_features.columns
    if any(c == raw or c.startswith(raw + "_") for raw in DROP_RAW_FAMILIES)
]
model_df = customer_features.drop(columns=drop_cols)
del customer_features
gc.collect()
print(f"Sparse-family exclusion: removed {len(drop_cols)} columns -> model_df {model_df.shape}")


Loaded (458913, 1301) in 4.7s. RSS: 1.49 GB


Sparse-family exclusion: removed 60 columns -> model_df (458913, 1241)


In [3]:
TRUE_CATEGORICAL_RAW = ["D_63", "D_64"]
CANDIDATE_CODE_RAW = ["D_116", "D_114", "D_66", "B_31", "D_120", "B_30", "D_126", "B_38", "D_117", "D_68"]
CATEGORICAL_RAW = TRUE_CATEGORICAL_RAW + CANDIDATE_CODE_RAW  # 12, same as Phase 3/4

categorical_cols = [f"{c}_first" for c in CATEGORICAL_RAW] + [f"{c}_last" for c in CATEGORICAL_RAW]
exclude_cols = ["customer_ID", "target"]
feature_cols = [c for c in model_df.columns if c not in exclude_cols]
assert len(feature_cols) == 1239, "Full feature count no longer matches Phase 3/4 -- stop."
print(f"Total modeling features (matches Phase 3/4): {len(feature_cols)}")


Total modeling features (matches Phase 3/4): 1239


In [4]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    model_df, test_size=0.20, stratify=model_df["target"], random_state=RANDOM_SEED,
)
del model_df
gc.collect()

print(f"Train customers:      {len(train_df):,}  (Phase 3/4: 367,130)")
print(f"Validation customers: {len(val_df):,}  (Phase 3/4: 91,783)")
print(f"Train default rate:      {train_df['target'].mean():.4f}  (Phase 3/4: 0.2589)")
print(f"Validation default rate: {val_df['target'].mean():.4f}  (Phase 3/4: 0.2589)")

assert len(train_df) == 367_130 and len(val_df) == 91_783, "Split does not match Phase 3/4 -- stop."
print("\nSplit matches Phase 3/4 exactly. Not a temporal/out-of-time split -- same stratified random holdout as before.")


Train customers:      367,130  (Phase 3/4: 367,130)
Validation customers: 91,783  (Phase 3/4: 91,783)
Train default rate:      0.2589  (Phase 3/4: 0.2589)
Validation default rate: 0.2589  (Phase 3/4: 0.2589)

Split matches Phase 3/4 exactly. Not a temporal/out-of-time split -- same stratified random holdout as before.


In [5]:
def amex_metric(y_true, y_pred) -> dict:
    """Official AMEX competition metric -- identical to Phase 3/4's verified implementation."""
    df = pd.DataFrame({"target": np.asarray(y_true), "prediction": np.asarray(y_pred)})
    df = df.sort_values("prediction", ascending=False).reset_index(drop=True)
    df["weight"] = np.where(df["target"] == 0, 20, 1)

    four_pct_cutoff = int(0.04 * df["weight"].sum())
    df["weight_cumsum"] = df["weight"].cumsum()
    df_cutoff = df.loc[df["weight_cumsum"] <= four_pct_cutoff]
    top4_capture = (df_cutoff["target"] == 1).sum() / (df["target"] == 1).sum()

    def weighted_gini(frame: pd.DataFrame) -> float:
        f = frame.sort_values("prediction", ascending=False)
        f_random = (f["weight"] / f["weight"].sum()).cumsum()
        total_pos = (f["target"] * f["weight"]).sum()
        cum_pos_found = (f["target"] * f["weight"]).cumsum()
        lorentz = cum_pos_found / total_pos
        return ((lorentz - f_random) * f["weight"]).sum()

    perfect = df.copy()
    perfect["prediction"] = perfect["target"]
    normalized_gini = weighted_gini(df) / weighted_gini(perfect)

    m = 0.5 * (normalized_gini + top4_capture)
    return {"amex_metric": m, "normalized_gini": normalized_gini, "top4pct_capture": top4_capture}

from sklearn.metrics import roc_auc_score
print("amex_metric loaded (identical to Phase 3/4, sanity-checked there).")


amex_metric loaded (identical to Phase 3/4, sanity-checked there).


In [6]:
def clean_categorical(series: pd.Series) -> pd.Series:
    return series.astype(object).where(series.notna(), "MISSING").astype(str)

for c in categorical_cols:
    train_df[c] = clean_categorical(train_df[c])
    val_df[c] = clean_categorical(val_df[c])
print(f"Cleaned {len(categorical_cols)} categorical columns (NaN -> 'MISSING') in train_df/val_df.")


Cleaned 24 categorical columns (NaN -> 'MISSING') in train_df/val_df.


## 1. Reconstruct the Phase 4 852-Feature Configuration

Rebuilt from the same feature-group classifier and group logic Phase 4 used to define
"Latest + Historical" — not a hand-typed list of 852 column names. Verified against Phase
4's reported count and composition before any model touches it.


In [7]:
def classify_feature(col: str) -> str:
    if col in ("statement_count", "history_length_days"):
        return "history"
    if col.endswith("_missing_rate"):
        return "missing_rate"
    if col.endswith("_nunique"):
        return "nunique"
    if col in categorical_cols:
        if col.endswith("_first"):
            return "categorical_first"
        if col.endswith("_last"):
            return "categorical_last"
    for suffix in ["_mean", "_std", "_min", "_max", "_change", "_first", "_last"]:
        if col.endswith(suffix):
            return suffix[1:]
    return "UNCLASSIFIED"

feature_group = {c: classify_feature(c) for c in feature_cols}
assert all(g != "UNCLASSIFIED" for g in feature_group.values()), "Unclassified feature found -- stop."

# Identical to Phase 4's EXPERIMENT_GROUPS["C: Latest + historical"]
LATEST_HISTORICAL_GROUPS = ["last", "categorical_last", "mean", "std", "min", "max"]
lh_feature_cols = [c for c, g in feature_group.items() if g in LATEST_HISTORICAL_GROUPS]
lh_categorical_cols = [c for c in lh_feature_cols if c in categorical_cols]

group_counts = pd.Series(feature_group).loc[lambda s: s.index.isin(lh_feature_cols)].value_counts()
print("Composition of the reconstructed feature set:")
print(group_counts)
print(f"\nTotal features: {len(lh_feature_cols)} (Phase 4 reported: 852)")
print(f"Categorical features within it: {len(lh_categorical_cols)} (Phase 4: 12)")

assert len(lh_feature_cols) == 852, f"Expected 852 features, got {len(lh_feature_cols)} -- stop."
assert len(lh_categorical_cols) == 12
assert set(group_counts.index) == {"last", "categorical_last", "mean", "std", "min", "max"}
print("\nMatches Phase 4's 852-feature 'Latest + Historical' configuration exactly.")


Composition of the reconstructed feature set:
mean                168
std                 168
min                 168
max                 168
last                168
categorical_last     12
Name: count, dtype: int64

Total features: 852 (Phase 4 reported: 852)
Categorical features within it: 12 (Phase 4: 12)

Matches Phase 4's 852-feature 'Latest + Historical' configuration exactly.


## 2. Phase 5 Baselines (Cited from Phase 4, Not Retrained)

Same 852-feature configuration, same split, same hyperparameters as Phase 4's Experiment C
and its CatBoost confirmation — re-running either would reproduce the same numbers Phase 4
already measured (Phase 4 verified this kind of exact reproducibility for its own baseline
check). Cited directly from the executed Phase 4 notebook:


In [8]:
# Full precision as reported in Phase 4's executed comparison table (Section 4) and
# CatBoost confirmation output (Section 10) -- transcribed, not re-measured.
PHASE4_LGBM_BASELINE = {
    "roc_auc": 0.962040, "normalized_gini": 0.924079, "top4pct_capture": 0.662838,
    "amex_metric": 0.793458, "best_iteration": 498, "train_time": 95.574633,
}
PHASE4_CATBOOST_BASELINE = {
    "roc_auc": 0.9623, "normalized_gini": 0.9247, "top4pct_capture": 0.6631,
    "amex_metric": 0.7939, "best_iteration": 1530, "train_time": 634.1,
}
print("LightGBM baseline (Phase 4, Experiment C):", PHASE4_LGBM_BASELINE)
print("CatBoost baseline (Phase 4, confirmation run):", PHASE4_CATBOOST_BASELINE)
print("\n(CatBoost figures carry 4-decimal precision -- that's the full precision Phase 4 reported;")
print(" no higher-precision source exists without retraining, which the baseline citation is meant to avoid.)")


LightGBM baseline (Phase 4, Experiment C): {'roc_auc': 0.96204, 'normalized_gini': 0.924079, 'top4pct_capture': 0.662838, 'amex_metric': 0.793458, 'best_iteration': 498, 'train_time': 95.574633}
CatBoost baseline (Phase 4, confirmation run): {'roc_auc': 0.9623, 'normalized_gini': 0.9247, 'top4pct_capture': 0.6631, 'amex_metric': 0.7939, 'best_iteration': 1530, 'train_time': 634.1}

(CatBoost figures carry 4-decimal precision -- that's the full precision Phase 4 reported;
 no higher-precision source exists without retraining, which the baseline citation is meant to avoid.)


## 3–5. LightGBM Optimization

**A finding worth surfacing before tuning anything:** Phase 3/4's baseline set
`subsample=0.8` intending row-level stochastic regularization, but LightGBM's own
documentation is explicit that `subsample` (`bagging_fraction`) only takes effect when
`subsample_freq` (`bagging_freq`) is set to a positive value — `subsample_freq` defaults to
**0**, which means it was never enabled: `subsample_freq : int, optional (default=0) —
Frequency of subsample, <=0 means no enable.` Every LightGBM run in Phases 3 and 4 used
active column subsampling (`colsample_bytree=0.8`, which has no such requirement) but
**inactive** row subsampling. This is treated as its own targeted experiment below (make
`subsample_freq` actually do something) rather than silently fixed and re-baselined —
whether activating it helps or not is itself useful information.

**Design:** every experiment changes the baseline by exactly one dimension (except the
combined-configuration step, which is explicitly a combination), so each result is
individually interpretable. `max_depth` is not tuned separately from `num_leaves` — with
leaf-wise growth already bounded by `num_leaves`, an independent depth limit is a mostly
redundant lever for this model.

**A note on how these results were obtained:** this notebook's first full execution ran
all 7 of the LightGBM experiments below successfully, then was interrupted by the
environment partway through the second new CatBoost experiment (documented in Section 14).
The 7 LightGBM results are real, valid, and complete — re-running an already-completed,
deterministic experiment identically would burn compute for no new information, so 6 of
them are cited here exactly as originally measured, and only the winning configuration is
re-run (needed anyway to obtain fresh prediction arrays for Phase 6, since the interrupted
kernel's in-memory predictions were lost with it).


In [9]:
BASE_LGBM_PARAMS = dict(
    objective="binary", n_estimators=2000, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED, n_jobs=-1, verbosity=-1,
)  # identical to Phase 3/4's baseline, including the inert subsample_freq=0 default

MEANINGFUL_THRESHOLD = 0.0005  # differences below this are treated as noise, not signal,
                                 # consistent with the instruction that 0.0001-0.0003 is not "major"

def run_lgbm_experiment(name: str, param_overrides: dict, early_stopping_rounds: int = 100) -> dict:
    import lightgbm as lgb

    train_x = train_df[lh_feature_cols].copy()
    val_x = val_df[lh_feature_cols].copy()
    for c in lh_categorical_cols:
        train_categories = pd.unique(train_x[c])
        train_x[c] = pd.Categorical(train_x[c], categories=train_categories)
        val_x[c] = pd.Categorical(val_x[c], categories=train_categories)

    params = {**BASE_LGBM_PARAMS, **param_overrides}
    lgbm = lgb.LGBMClassifier(**params)

    t0 = time.time()
    lgbm.fit(
        train_x, train_df["target"],
        eval_X=val_x, eval_y=val_df["target"],
        eval_metric="auc",
        categorical_feature=lh_categorical_cols,
        callbacks=[lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=False), lgb.log_evaluation(period=0)],
    )
    train_time = time.time() - t0
    best_iter = lgbm.best_iteration_
    pred = lgbm.predict_proba(val_x, num_iteration=best_iter)[:, 1]

    auc = roc_auc_score(val_df["target"], pred)
    amex = amex_metric(val_df["target"].values, pred)
    delta = amex["amex_metric"] - PHASE4_LGBM_BASELINE["amex_metric"]

    del train_x, val_x
    gc.collect()

    result = {
        "name": name, "params_changed": param_overrides, "roc_auc": auc,
        "normalized_gini": amex["normalized_gini"], "top4pct_capture": amex["top4pct_capture"],
        "amex_metric": amex["amex_metric"], "delta_vs_baseline": delta,
        "train_time": train_time, "best_iteration": best_iter, "model": lgbm, "pred": pred,
    }
    print(f"[{name}] changed={param_overrides} AUC={auc:.4f} AMEX={amex['amex_metric']:.4f} "
          f"(delta {delta:+.4f}) time={train_time:.1f}s best_iter={best_iter}. RSS: {rss_gb():.2f} GB")
    return result


### Experiments 1–4 and 6–7 — Cited from the Interrupted Run

Transcribed exactly from that run's printed output (Experiments 1–4: complexity and
regularization variants; Experiment 6: combined configuration; Experiment 7: lower
learning rate refinement). No `model`/`pred` objects are kept for these — only the
winning configuration needs real predictions, obtained fresh below.


In [10]:
# Real results from the interrupted run's LightGBM section (Experiments 1-4, 6-7).
CITED_LGBM_RESULTS = {
    "Lower complexity (num_leaves=15)": {
        "name": "Lower complexity (num_leaves=15)", "params_changed": {"num_leaves": 15}, "roc_auc": 0.9621, "normalized_gini": 0.9242,
        "top4pct_capture": 0.6587, "amex_metric": 0.7915, "delta_vs_baseline": -0.0020,
        "train_time": 143.578, "best_iteration": 1499,
    },
    "Higher complexity (num_leaves=63)": {
        "name": "Higher complexity (num_leaves=63)", "params_changed": {"num_leaves": 63}, "roc_auc": 0.9620, "normalized_gini": 0.9240,
        "top4pct_capture": 0.6598, "amex_metric": 0.7919, "delta_vs_baseline": -0.0016,
        "train_time": 138.3727, "best_iteration": 605,
    },
    "Stronger min-leaf reg (min_child_samples=100)": {
        "name": "Stronger min-leaf reg (min_child_samples=100)", "params_changed": {"min_child_samples": 100}, "roc_auc": 0.9622, "normalized_gini": 0.9245,
        "top4pct_capture": 0.6598, "amex_metric": 0.7921, "delta_vs_baseline": -0.0013,
        "train_time": 141.0294, "best_iteration": 802,
    },
    "Activate row subsampling (subsample_freq=1)": {
        "name": "Activate row subsampling (subsample_freq=1)", "params_changed": {"subsample_freq": 1}, "roc_auc": 0.9620, "normalized_gini": 0.9241,
        "top4pct_capture": 0.6641, "amex_metric": 0.7941, "delta_vs_baseline": 0.0006,
        "train_time": 106.8037, "best_iteration": 537,
    },
    "Combined (best individual changes)": {
        "name": "Combined (best individual changes)", "params_changed": {"subsample_freq": 1, "reg_alpha": 1.0, "reg_lambda": 1.0},
        "roc_auc": 0.9624, "normalized_gini": 0.9248, "top4pct_capture": 0.6646,
        "amex_metric": 0.7947, "delta_vs_baseline": 0.0013, "train_time": 156.5493, "best_iteration": 766,
    },
    "Lower LR refinement (lr=0.03)": {
        "name": "Lower LR refinement (lr=0.03)", "params_changed": {"subsample_freq": 1, "reg_alpha": 1.0, "reg_lambda": 1.0,
                            "learning_rate": 0.03, "n_estimators": 4000},
        "roc_auc": 0.9624, "normalized_gini": 0.9249, "top4pct_capture": 0.6650,
        "amex_metric": 0.7949, "delta_vs_baseline": 0.0015, "train_time": 226.5307, "best_iteration": 1059,
    },
}
lgbm_results = dict(CITED_LGBM_RESULTS)  # model/pred added below for the winner only
beneficial_overrides = {"subsample_freq": 1, "reg_alpha": 1.0, "reg_lambda": 1.0}  # from the interrupted run
ran_lr_refinement = True  # it was run in the interrupted session; cited above
print(f"Cited {len(CITED_LGBM_RESULTS)} LightGBM results from the interrupted run.")
for name, r in CITED_LGBM_RESULTS.items():
    print(f"  [{name}] AMEX={r['amex_metric']:.4f} (delta {r['delta_vs_baseline']:+.4f})")


Cited 6 LightGBM results from the interrupted run.
  [Lower complexity (num_leaves=15)] AMEX=0.7915 (delta -0.0020)
  [Higher complexity (num_leaves=63)] AMEX=0.7919 (delta -0.0016)
  [Stronger min-leaf reg (min_child_samples=100)] AMEX=0.7921 (delta -0.0013)
  [Activate row subsampling (subsample_freq=1)] AMEX=0.7941 (delta +0.0006)
  [Combined (best individual changes)] AMEX=0.7947 (delta +0.0013)
  [Lower LR refinement (lr=0.03)] AMEX=0.7949 (delta +0.0015)


### Experiment 5 — L1/L2 Regularization (`reg_alpha=1.0, reg_lambda=1.0`) — Re-run for Real Predictions

This was the best-performing configuration in the interrupted run (AMEX 0.7952,
delta +0.0018). Re-run here fresh — both to obtain real prediction arrays for Phase 6, and
as a reproducibility check against the cited figure (LightGBM is deterministic with a fixed
seed, so these should match closely).


In [11]:
lgbm_results["L1/L2 regularization (alpha=1.0, lambda=1.0)"] = run_lgbm_experiment(
    "L1/L2 regularization (alpha=1.0, lambda=1.0)", {"reg_alpha": 1.0, "reg_lambda": 1.0}
)
cited_l1l2_amex = 0.7952
fresh_l1l2_amex = lgbm_results["L1/L2 regularization (alpha=1.0, lambda=1.0)"]["amex_metric"]
print(f"\nReproducibility check: cited AMEX {cited_l1l2_amex:.4f} vs. fresh AMEX {fresh_l1l2_amex:.4f} "
      f"(diff {fresh_l1l2_amex - cited_l1l2_amex:+.4f}, expected ~0)")


[L1/L2 regularization (alpha=1.0, lambda=1.0)] changed={'reg_alpha': 1.0, 'reg_lambda': 1.0} AUC=0.9623 AMEX=0.7952 (delta +0.0018) time=172.6s best_iter=924. RSS: 0.58 GB

Reproducibility check: cited AMEX 0.7952 vs. fresh AMEX 0.7952 (diff +0.0000, expected ~0)


## 6. Best LightGBM Configuration

All LightGBM experiments actually run, ranked by AMEX metric (the primary objective, not
AUC alone).


In [12]:
lgbm_table = pd.DataFrame([
    {"Experiment": name, "Params Changed": str(r["params_changed"]), "ROC AUC": r["roc_auc"],
     "Normalized Gini": r["normalized_gini"], "Top-4% Capture": r["top4pct_capture"],
     "AMEX Metric": r["amex_metric"], "Delta vs. Baseline": r["delta_vs_baseline"],
     "Train Time (s)": r["train_time"], "Best Iteration": r["best_iteration"]}
    for name, r in lgbm_results.items()
]).sort_values("AMEX Metric", ascending=False).reset_index(drop=True)

print(f"LightGBM experiments run: {len(lgbm_results)}")
lgbm_table.round(4)


LightGBM experiments run: 7


,Experiment,Params Changed,ROC AUC,Normalized Gini,Top-4% Capture,AMEX Metric,Delta vs. Baseline,Train Time (s),Best Iteration
0,"L1/L2 regularization (alpha=1.0, lambda=1.0)","{'reg_alpha': 1.0, 'reg_lambda': 1.0}",0.9623,0.9246,0.6658,0.7952,0.0018,172.5799,924
1,Lower LR refinement (lr=0.03),"{'subsample_freq': 1, 'reg_alpha': 1.0, 'reg_l...",0.9624,0.9249,0.6650,0.7949,0.0015,226.5307,1059
2,Combined (best individual changes),"{'subsample_freq': 1, 'reg_alpha': 1.0, 'reg_l...",0.9624,0.9248,0.6646,0.7947,0.0013,156.5493,766
3,Activate row subsampling (subsample_freq=1),{'subsample_freq': 1},0.9620,0.9241,0.6641,0.7941,0.0006,106.8037,537
4,Stronger min-leaf reg (min_child_samples=100),{'min_child_samples': 100},0.9622,0.9245,0.6598,0.7921,-0.0013,141.0294,802
5,Higher complexity (num_leaves=63),{'num_leaves': 63},0.9620,0.9240,0.6598,0.7919,-0.0016,138.3727,605
6,Lower complexity (num_leaves=15),{'num_leaves': 15},0.9621,0.9242,0.6587,0.7915,-0.0020,143.5780,1499


In [13]:
# The best configuration might be a new tuned experiment, or it might be that nothing beat
# the cited Phase 4 baseline -- both are checked explicitly rather than assuming tuning won.
best_tuned_name = lgbm_table.iloc[0]["Experiment"]
best_tuned = lgbm_results[best_tuned_name]

reproduced_lgbm_baseline = False
if best_tuned["amex_metric"] > PHASE4_LGBM_BASELINE["amex_metric"]:
    best_lgbm_name = best_tuned_name
    best_lgbm = best_tuned
    print(f"Best LightGBM configuration: {best_lgbm_name} (beats the Phase 4 baseline)")
else:
    print("No tuned experiment beat the Phase 4 852-feature baseline -- the baseline itself")
    print("remains the best LightGBM configuration. Reproducing it once now (not counted against")
    print("the tuning budget) solely to obtain real validation predictions for Phase 6.")
    best_lgbm_name = "Phase 4 baseline (reproduced for predictions)"
    best_lgbm = run_lgbm_experiment(best_lgbm_name, {})
    reproduced_lgbm_baseline = True
    reproduction_gap = best_lgbm["amex_metric"] - PHASE4_LGBM_BASELINE["amex_metric"]
    print(f"Reproduction check: fresh AMEX {best_lgbm['amex_metric']:.4f} vs. Phase 4's cited "
          f"{PHASE4_LGBM_BASELINE['amex_metric']:.4f} (diff {reproduction_gap:+.4f}, expected ~0)")
    print(f"\nBest LightGBM configuration: {best_lgbm_name}")

lgbm_gain = best_lgbm["amex_metric"] - PHASE4_LGBM_BASELINE["amex_metric"]
gini_gain = best_lgbm["normalized_gini"] - PHASE4_LGBM_BASELINE["normalized_gini"]
top4_gain = best_lgbm["top4pct_capture"] - PHASE4_LGBM_BASELINE["top4pct_capture"]

print(f"AMEX metric: {PHASE4_LGBM_BASELINE['amex_metric']:.4f} (Phase 4 baseline) -> {best_lgbm['amex_metric']:.4f} "
      f"({lgbm_gain:+.4f})")
print(f"  Normalized Gini component: {PHASE4_LGBM_BASELINE['normalized_gini']:.4f} -> {best_lgbm['normalized_gini']:.4f} ({gini_gain:+.4f})")
print(f"  Top-4% capture component:  {PHASE4_LGBM_BASELINE['top4pct_capture']:.4f} -> {best_lgbm['top4pct_capture']:.4f} ({top4_gain:+.4f})")
verdict = "meaningful" if abs(lgbm_gain) >= MEANINGFUL_THRESHOLD else "within noise for a single holdout split"
print(f"\nThis improvement is treated as {verdict} (threshold: {MEANINGFUL_THRESHOLD}).")


Best LightGBM configuration: L1/L2 regularization (alpha=1.0, lambda=1.0) (beats the Phase 4 baseline)
AMEX metric: 0.7935 (Phase 4 baseline) -> 0.7952 (+0.0018)
  Normalized Gini component: 0.9241 -> 0.9246 (+0.0005)
  Top-4% capture component:  0.6628 -> 0.6658 (+0.0030)

This improvement is treated as meaningful (threshold: 0.0005).


## 7–9. CatBoost Optimization — Documented Technical Issue

**CatBoost tuning could not be completed in this environment, despite ten attempts across
two approaches.** This is reported transparently rather than hidden or worked around with
fabricated numbers, per the project's own standard for handling technical failures.

**What was tried:** running CatBoost training (a) inline inside this notebook via the same
`nbclient`-driven execution used for every other phase, and (b) as standalone Python
scripts outside any notebook, to rule out a Jupyter-specific cause. Every configuration
that changed **any** hyperparameter away from the exact previously-successful baseline
(`iterations=2000, depth=6, learning_rate=0.05`, CatBoost's other defaults) was killed by
the environment during `.fit()` — three attempts at `depth=8 + l2_leaf_reg=6.0`, two
attempts at `learning_rate=0.03 + iterations=2500`, both inline and standalone, all killed
with no error message, typically within 1–2 minutes of training starting. In contrast, the
**exact baseline configuration succeeded cleanly twice in a row**, standalone, in 856.6s
and (in an earlier session) 634.1s, both matching Phase 4's cited baseline almost exactly.
System-level available memory was consistently healthy (2.5–3.9GB free) at every failure,
ruling out simple memory exhaustion as the cause; the pattern instead suggests some
resource-allocation instability specific to non-baseline CatBoost configurations on this
852-feature, 367,130-row dataset in this particular environment — not a code defect (the
same code succeeds for the unchanged baseline) and not explained by AMEX-metric-adjacent
reasoning at all.

**One real data point exists from CatBoost tuning**, captured in this notebook's very
first execution attempt before a later, unrelated interruption: `depth=8 + l2_leaf_reg=6.0`
completed successfully that one time, scoring AUC 0.9625, AMEX metric 0.7959 (delta +0.0020
vs. the 0.7939 baseline), in 1127.3s. It could not be reproduced in three further attempts,
and its Gini/top-4%-capture breakdown and validation predictions were never captured (lost
when that session later failed). **This is treated as an unconfirmed, single-observation
signal — not a validated Phase 5 finding** — reported below for completeness, not used to
select the "best" CatBoost configuration.

**What Phase 5 actually reports for CatBoost:** the baseline configuration, reproduced
twice with real, matching numbers and real validation predictions saved to disk — CatBoost
tuning is reported as *attempted but not completed*, and the carried-forward CatBoost
figure is Phase 4's baseline (0.7939), unchanged.


In [14]:
BASE_CB_PARAMS = dict(
    iterations=2000, learning_rate=0.05, depth=6,
    loss_function="Logloss", eval_metric="AUC",
    random_seed=RANDOM_SEED, early_stopping_rounds=100, verbose=False,
)  # identical to Phase 3/4's baseline -- the only CatBoost configuration that ran reliably

# The one real, unconfirmed tuning data point (see markdown above) -- transcribed exactly
# from this notebook's first execution attempt. No model/pred available; not selected as
# the "best" configuration because it could not be reproduced or completed.
UNCONFIRMED_CB_TUNING_RESULT = {
    "name": "Depth 8 + stronger L2 (l2_leaf_reg=6.0) [UNCONFIRMED, single observation]",
    "params_changed": {"depth": 8, "l2_leaf_reg": 6.0},
    "roc_auc": 0.9625, "amex_metric": 0.7959, "delta_vs_baseline": 0.0020,
    "train_time": 1127.3, "best_iteration": 1306, "ceiling": 2000,
    "normalized_gini": None, "top4pct_capture": None,  # never captured -- see markdown
}
print("Unconfirmed CatBoost tuning signal (not used for model selection):")
print(f"  {UNCONFIRMED_CB_TUNING_RESULT['name']}: AMEX {UNCONFIRMED_CB_TUNING_RESULT['amex_metric']} "
      f"(delta {UNCONFIRMED_CB_TUNING_RESULT['delta_vs_baseline']:+.4f}), reproduced 0/3 further attempts.")

catboost_results = {}  # intentionally empty -- no CatBoost tuning experiment completed reliably enough to count
ran_cb_combined = False


Unconfirmed CatBoost tuning signal (not used for model selection):
  Depth 8 + stronger L2 (l2_leaf_reg=6.0) [UNCONFIRMED, single observation]: AMEX 0.7959 (delta +0.0020), reproduced 0/3 further attempts.


## CatBoost Baseline (Reproduced Standalone, Real Predictions Saved)

Run as a standalone script (`run_catboost_baseline_only.py`, outside this notebook, given
the instability documented above), using the identical 852-feature setup, split, and
baseline hyperparameters constructed the same way as everywhere else in this notebook. Its
saved JSON result and prediction file are loaded here, not recomputed inline.


In [15]:
import json

CB_STANDALONE_JSON = "/private/tmp/claude-501/-Users-athena-Desktop-amex-default-prediction/e00064d7-7f68-483c-a37d-0e500ec9a4c9/scratchpad/catboost_baseline_result.json"
with open(CB_STANDALONE_JSON) as f:
    best_cb = json.load(f)

# Cross-check against Phase 4's cited baseline -- should match closely (same config, same split, same seed).
reproduced_cb_baseline = True
reproduction_gap = best_cb["amex_metric"] - PHASE4_CATBOOST_BASELINE["amex_metric"]
print(f"Loaded standalone baseline result: {best_cb['name']}")
print(f"  AUC={best_cb['roc_auc']:.4f} AMEX={best_cb['amex_metric']:.4f} "
      f"best_iter={best_cb['best_iteration']} train_time={best_cb['train_time']:.1f}s")
print(f"Reproduction check vs. Phase 4's cited baseline: diff {reproduction_gap:+.4f} (expected ~0)")

best_cb_name = best_cb["name"]

CB_PRED_PATH = os.path.join(PRED_DIR, "phase5_catboost_val_predictions.parquet")
cb_pred_df_loaded = pd.read_parquet(CB_PRED_PATH)
best_cb["pred"] = cb_pred_df_loaded["prediction"].values
print(f"Loaded {len(best_cb['pred']):,} real validation predictions from {CB_PRED_PATH}")

cb_gain = best_cb["amex_metric"] - PHASE4_CATBOOST_BASELINE["amex_metric"]
cb_gini_gain = best_cb["normalized_gini"] - PHASE4_CATBOOST_BASELINE["normalized_gini"]
cb_top4_gain = best_cb["top4pct_capture"] - PHASE4_CATBOOST_BASELINE["top4pct_capture"]
print(f"\nAMEX metric: {PHASE4_CATBOOST_BASELINE['amex_metric']:.4f} (Phase 4 baseline) -> {best_cb['amex_metric']:.4f} ({cb_gain:+.4f})")
print("No CatBoost tuning improvement is claimed -- Phase 5's CatBoost result equals Phase 4's baseline.")


Loaded standalone baseline result: Phase 4 baseline (reproduced standalone for predictions)
  AUC=0.9623 AMEX=0.7939 best_iter=1530 train_time=856.6s
Reproduction check vs. Phase 4's cited baseline: diff +0.0000 (expected ~0)
Loaded 91,783 real validation predictions from ../data/processed/phase5_catboost_val_predictions.parquet

AMEX metric: 0.7939 (Phase 4 baseline) -> 0.7939 (+0.0000)
No CatBoost tuning improvement is claimed -- Phase 5's CatBoost result equals Phase 4's baseline.


## 10. Final Comparison: Baseline vs. Tuned


In [16]:
final_comparison = pd.DataFrame([
    {"Model": "LightGBM", "Configuration": "Phase 4 baseline", **PHASE4_LGBM_BASELINE, "Delta vs. Baseline": 0.0},
    {"Model": "LightGBM", "Configuration": best_lgbm_name, "roc_auc": best_lgbm["roc_auc"],
     "normalized_gini": best_lgbm["normalized_gini"], "top4pct_capture": best_lgbm["top4pct_capture"],
     "amex_metric": best_lgbm["amex_metric"], "best_iteration": best_lgbm["best_iteration"],
     "train_time": best_lgbm["train_time"], "Delta vs. Baseline": lgbm_gain},
    {"Model": "CatBoost", "Configuration": "Phase 4 baseline", **PHASE4_CATBOOST_BASELINE, "Delta vs. Baseline": 0.0},
    {"Model": "CatBoost", "Configuration": best_cb_name, "roc_auc": best_cb["roc_auc"],
     "normalized_gini": best_cb["normalized_gini"], "top4pct_capture": best_cb["top4pct_capture"],
     "amex_metric": best_cb["amex_metric"], "best_iteration": best_cb["best_iteration"],
     "train_time": best_cb["train_time"], "Delta vs. Baseline": cb_gain},
])[["Model", "Configuration", "roc_auc", "normalized_gini", "top4pct_capture", "amex_metric", "Delta vs. Baseline", "train_time", "best_iteration"]]
final_comparison.columns = ["Model", "Configuration", "ROC AUC", "Normalized Gini", "Top-4% Capture", "AMEX Metric", "Delta vs. Baseline", "Train Time (s)", "Best Iteration"]
final_comparison.round(4)


,Model,Configuration,ROC AUC,Normalized Gini,Top-4% Capture,AMEX Metric,Delta vs. Baseline,Train Time (s),Best Iteration
0,LightGBM,Phase 4 baseline,0.9620,0.9241,0.6628,0.7935,0.0000,95.5746,498
1,LightGBM,"L1/L2 regularization (alpha=1.0, lambda=1.0)",0.9623,0.9246,0.6658,0.7952,0.0018,172.5799,924
2,CatBoost,Phase 4 baseline,0.9623,0.9247,0.6631,0.7939,0.0000,634.1000,1530
3,CatBoost,Phase 4 baseline (reproduced standalone for pr...,0.9623,0.9247,0.6631,0.7939,0.0000,856.6130,1530


## 11. Did Tuning Materially Help?


In [17]:
print("=" * 78)
print("OPTIMIZATION VALUE (all figures from executed experiments/citations above)")
print("=" * 78)
print(f"""
LightGBM: {PHASE4_LGBM_BASELINE['amex_metric']:.4f} -> {best_lgbm['amex_metric']:.4f} ({lgbm_gain:+.4f}) -- {"meaningful" if abs(lgbm_gain) >= MEANINGFUL_THRESHOLD else "within noise, not a major improvement"}.
  Driven by: Gini {gini_gain:+.4f}, Top-4% capture {top4_gain:+.4f}
  ({"primarily Gini" if abs(gini_gain) > abs(top4_gain)*1.5 else "primarily top-4% capture" if abs(top4_gain) > abs(gini_gain)*1.5 else "both components roughly equally"}).

CatBoost: {PHASE4_CATBOOST_BASELINE['amex_metric']:.4f} -> {best_cb['amex_metric']:.4f} ({cb_gain:+.4f}) -- {"meaningful" if abs(cb_gain) >= MEANINGFUL_THRESHOLD else "within noise, not a major improvement"}.
  Driven by: Gini {cb_gini_gain:+.4f}, Top-4% capture {cb_top4_gain:+.4f}
  ({"primarily Gini" if abs(cb_gini_gain) > abs(cb_top4_gain)*1.5 else "primarily top-4% capture" if abs(cb_top4_gain) > abs(cb_gini_gain)*1.5 else "both components roughly equally"}).

More responsive to tuning: {"LightGBM" if abs(lgbm_gain) > abs(cb_gain) else "CatBoost" if abs(cb_gain) > abs(lgbm_gain) else "Neither -- comparable (small) movement"}.
""")

positive_lgbm = [r["name"] for r in lgbm_results.values() if r.get("delta_vs_baseline", 0) > MEANINGFUL_THRESHOLD]
negative_lgbm = [r["name"] for r in lgbm_results.values() if r.get("delta_vs_baseline", 0) < -MEANINGFUL_THRESHOLD]
print(f"LightGBM changes that helped (> {MEANINGFUL_THRESHOLD}): {positive_lgbm if positive_lgbm else 'none'}")
print(f"LightGBM changes that hurt (< -{MEANINGFUL_THRESHOLD}): {negative_lgbm if negative_lgbm else 'none'}")

positive_cb = [r["name"] for r in catboost_results.values() if r.get("delta_vs_baseline", 0) > MEANINGFUL_THRESHOLD]
negative_cb = [r["name"] for r in catboost_results.values() if r.get("delta_vs_baseline", 0) < -MEANINGFUL_THRESHOLD]
print(f"CatBoost changes that helped (> {MEANINGFUL_THRESHOLD}): {positive_cb if positive_cb else 'none (no experiment completed reliably -- see Sections 7-9)'}")
print(f"CatBoost changes that hurt (< -{MEANINGFUL_THRESHOLD}): {negative_cb if negative_cb else 'none'}")
print(f"CatBoost unconfirmed signal (single observation, not reproduced): "
      f"{UNCONFIRMED_CB_TUNING_RESULT['name']} at delta {UNCONFIRMED_CB_TUNING_RESULT['delta_vs_baseline']:+.4f} "
      "-- suggestive of depth/regularization as a promising direction, but not validated.")


OPTIMIZATION VALUE (all figures from executed experiments/citations above)

LightGBM: 0.7935 -> 0.7952 (+0.0018) -- meaningful.
  Driven by: Gini +0.0005, Top-4% capture +0.0030
  (primarily top-4% capture).

CatBoost: 0.7939 -> 0.7939 (+0.0000) -- within noise, not a major improvement.
  Driven by: Gini -0.0000, Top-4% capture +0.0000
  (primarily top-4% capture).

More responsive to tuning: LightGBM.

LightGBM changes that helped (> 0.0005): ['Activate row subsampling (subsample_freq=1)', 'Combined (best individual changes)', 'Lower LR refinement (lr=0.03)', 'L1/L2 regularization (alpha=1.0, lambda=1.0)']
LightGBM changes that hurt (< -0.0005): ['Lower complexity (num_leaves=15)', 'Higher complexity (num_leaves=63)', 'Stronger min-leaf reg (min_child_samples=100)']
CatBoost changes that helped (> 0.0005): none (no experiment completed reliably -- see Sections 7-9)
CatBoost changes that hurt (< -0.0005): none
CatBoost unconfirmed signal (single observation, not reproduced): Depth 8 + 

## 12. Overfitting Awareness

Every experiment in this notebook — and in Phases 3 and 4 before it — was scored against
the **same** 91,783-customer validation split. Repeatedly optimizing against one fixed
holdout carries a real risk: some of any observed improvement could reflect this specific
split's noise rather than a genuinely better model. This is exactly why the experiment
budget here was capped (6–8 LightGBM, 2–3 CatBoost) rather than run until something looked
better — searching long enough against a fixed holdout will eventually produce an
improvement by chance alone, and that is not the same thing as a real one.

**Robust directional findings** (consistent with how the changes are expected to behave,
not just a single lucky number): the specific parameter values printed as "helped"/"hurt"
above. **Tiny score differences** — anything below the 0.0005 threshold
used throughout this notebook — are reported but explicitly not claimed as evidence that
one configuration is universally better; they are as likely to be holdout noise as real
signal, and would need repeated/cross-validated evaluation to distinguish.


## 13. Save Validation Predictions for Phase 6

Predictions from the **final selected** LightGBM and CatBoost configurations only — no
blending, no ensemble weighting. That comparison is explicitly Phase 6's job, using the
files saved here. Written under `data/processed/`, already excluded from version control by
`.gitignore`.


In [18]:
os.makedirs(PRED_DIR, exist_ok=True)

lgbm_pred_df = pd.DataFrame({
    "customer_ID": val_df["customer_ID"].values,
    "target": val_df["target"].values,
    "prediction": best_lgbm["pred"],
})
lgbm_pred_path = os.path.join(PRED_DIR, "phase5_lightgbm_val_predictions.parquet")
lgbm_pred_df.to_parquet(lgbm_pred_path, index=False)
print(f"Saved: {lgbm_pred_path} ({os.path.getsize(lgbm_pred_path)/1e6:.1f} MB) -- config: {best_lgbm_name}")

cb_pred_df = pd.DataFrame({
    "customer_ID": val_df["customer_ID"].values,
    "target": val_df["target"].values,
    "prediction": best_cb["pred"],
})
cb_pred_path = os.path.join(PRED_DIR, "phase5_catboost_val_predictions.parquet")
cb_pred_df.to_parquet(cb_pred_path, index=False)
print(f"Saved: {cb_pred_path} ({os.path.getsize(cb_pred_path)/1e6:.1f} MB) -- config: {best_cb_name}")


Saved: ../data/processed/phase5_lightgbm_val_predictions.parquet (7.0 MB) -- config: L1/L2 regularization (alpha=1.0, lambda=1.0)
Saved: ../data/processed/phase5_catboost_val_predictions.parquet (7.0 MB) -- config: Phase 4 baseline (reproduced standalone for predictions)


## 14. Resource Reporting


In [19]:
phase5_elapsed = time.time() - PHASE5_START
n_lgbm_run = len(lgbm_results)
n_cb_run_completed = 1  # baseline only -- see documented issue below

print("=== Documented technical issue: CatBoost tuning ===")
print("Across the full effort to complete Phase 5, CatBoost was attempted 10 times total:")
print("  - Attempt 1 (inline, this notebook): completed all 7 LightGBM experiments AND")
print("    CatBoost Experiment 1 (depth=8+L2, real result: AMEX 0.7959) successfully, then")
print("    was killed ~52 minutes in, partway through CatBoost Experiment 2.")
print("  - Attempts 2-4 (inline, this notebook, citing the 6 completed LightGBM results):")
print("    each killed within ~5-6 minutes, right as the next CatBoost experiment's .fit()")
print("    call started -- well before the ~52-minute mark, ruling out a simple long-runtime")
print("    ceiling as the sole explanation.")
print("  - Attempts 5-9 (standalone Python scripts, outside any notebook, to rule out a")
print("    Jupyter/nbclient-specific cause): the exact baseline configuration (depth=6,")
print("    iterations=2000) succeeded cleanly twice (634.1s and 856.6s); every non-baseline")
print("    configuration (depth=8 x3, lower-LR x1) was killed during .fit(), typically")
print("    within 1-2 minutes, with no error message, and with system memory consistently")
print("    healthy (2.5-3.9 GB available) at every failure.")
print("  - This (10th, final) execution: loads the standalone-verified baseline result and")
print("    real predictions from disk rather than attempting an 11th CatBoost tuning run.")
print()
print("Conclusion: this is a genuine, reproducible environment instability specific to")
print("non-baseline CatBoost configurations on this dataset -- not a memory shortage (memory")
print("was fine every time), not a code defect (the identical code succeeds for the baseline),")
print("and not something further retries were likely to resolve. Per the resource-management")
print("guidance for this phase, further CatBoost tuning attempts were stopped rather than")
print("continued indefinitely, and this state is reported here rather than hidden.")
print()
print(f"LightGBM experiments (real, either cited from the interrupted run or freshly re-run): {n_lgbm_run}")
print(f"CatBoost tuning experiments completed reliably enough to report: 0 of 2 planned")
print(f"CatBoost baseline reproductions completed: 2 (both standalone, both matching Phase 4 closely)")
print(f"Total runtime of this final, successful execution: {phase5_elapsed/60:.1f} minutes")
print(f"Final RSS: {rss_gb():.2f} GB")
print(f"Lower-LR LightGBM refinement run (cited from the interrupted run): {ran_lr_refinement}")
print(f"Combined CatBoost configuration run: {ran_cb_combined} (never reached -- see above)")
print(f"Phase 4 LightGBM baseline reproduced (for predictions only, not counted above): {reproduced_lgbm_baseline}")
print(f"Phase 4 CatBoost baseline reproduced (for predictions, via standalone script): {reproduced_cb_baseline}")
print()
print("All experiments that did complete ran strictly sequentially with train/validation")
print("copies released (del + gc.collect()) between them. No experiment was silently hidden --")
print("every attempted configuration, including the ones that could not be completed, is")
print("documented above rather than omitted.")


=== Documented technical issue: CatBoost tuning ===
Across the full effort to complete Phase 5, CatBoost was attempted 10 times total:
  - Attempt 1 (inline, this notebook): completed all 7 LightGBM experiments AND
    CatBoost Experiment 1 (depth=8+L2, real result: AMEX 0.7959) successfully, then
    was killed ~52 minutes in, partway through CatBoost Experiment 2.
  - Attempts 2-4 (inline, this notebook, citing the 6 completed LightGBM results):
    each killed within ~5-6 minutes, right as the next CatBoost experiment's .fit()
    call started -- well before the ~52-minute mark, ruling out a simple long-runtime
    ceiling as the sole explanation.
  - Attempts 5-9 (standalone Python scripts, outside any notebook, to rule out a
    Jupyter/nbclient-specific cause): the exact baseline configuration (depth=6,
    iterations=2000) succeeded cleanly twice (634.1s and 856.6s); every non-baseline
    configuration (depth=8 x3, lower-LR x1) was killed during .fit(), typically
    within 1-2

## Summary

Phase 5 adopted Phase 4's 852-feature "Latest + Historical" configuration (reconstructed
programmatically and verified, not hand-typed) and ran a compute-budgeted set of targeted
experiments against the unchanged Phase 3/4 validation split. **LightGBM tuning succeeded
completely**: 7 experiments, a confirmed best configuration (L1/L2 regularization, AMEX
0.7952, +0.0018 over baseline, driven mainly by top-4% capture), and real validation
predictions saved. **CatBoost tuning did not complete**, despite ten attempts across inline
and standalone approaches — documented transparently in Sections 7–9 and 14 rather than
worked around with fabricated numbers. CatBoost's baseline was reproduced twice, cleanly,
standalone, with real predictions saved; the carried-forward CatBoost figure for Phase 5 is
therefore Phase 4's baseline (0.7939), unchanged, alongside one unconfirmed single-
observation signal (depth=8 + stronger L2 → +0.0020) worth revisiting with more stable
compute rather than treated as validated. Predictions from LightGBM's confirmed winner and
CatBoost's reproduced baseline are both saved locally for Phase 6. No ensembling was
performed here, no hyperparameter search beyond the stated budget, and Logistic Regression
was not tuned.
